###Run Shared Libraries

In [0]:
%run ../Misc/SharedLibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimpurchitem"

###Read Bronze table

In [0]:
purchaseItemDf  = spark.table("bronze.purchitem")
purchaseItemDf .printSchema()

###Build Dimension/Fact table

In [0]:
dimPurchaseItemDf = purchaseItemDf.filter(purchaseItemDf.RecordId.isNotNull()
    ).select(
       purchaseItemDf.ItemId,
       F.trim(purchaseItemDf.Txt).alias("ProductName"),
       F.when(purchaseItemDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(purchaseItemDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
       F.from_utc_timestamp(purchaseItemDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
       F.when(purchaseItemDf.ValidFrom.isNull(), F.lit("1900-01-01")).otherwise(F.from_utc_timestamp(purchaseItemDf.ValidFrom,'CST')).cast("timestamp").alias("ValidFrom"),
       F.when(purchaseItemDf.ValidTo.isNull(), F.lit("1900-01-01")).otherwise(F.from_utc_timestamp(purchaseItemDf.ValidTo,'CST')).cast("timestamp").alias("ValidTo"),
       purchaseItemDf.Price.alias("ProductPerUnitCost"),
       purchaseItemDf.RecordId.alias("PurchItemRecordId"),
       purchaseItemDf.CategoryID.alias("CategoryID")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("PurchItemHashKey", F.xxhash64("PurchItemRecordId")
    )

display(dimPurchaseItemDf)

###Final dataframe

In [0]:
df_final = dimPurchaseItemDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)
